# Task 4: Fine-Tuning BERT on IMDB Dataset

In [1]:
import os
os.environ["WANDB_DISABLED"] = "true"

## Step 1: Install and Import Required Libraries

In [2]:
!pip install transformers==4.37.2 accelerate==0.27.2 peft==0.8.2 datasets==2.18.0 huggingface_hub==0.20.3 --no-deps -q

!pip install tokenizers==0.15.2 pyarrow-hotfix sentencepiece scikit-learn matplotlib -q

In [3]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## Step 2: Load IMDB Dataset

In [4]:
dataset = load_dataset("imdb")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


## Step 3: Load BERT Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

## Step 4: Tokenize Dataset

In [6]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

## Step 5: Create Smaller Training and Testing Samples

In [7]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(500))
small_test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(200))

## Step 6: Load Pretrained BERT Model

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step 7: Define Training Arguments

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"
)

## Step 8: Define Evaluation Metrics

In [10]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

## Step 9: Initialize Trainer

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_test_dataset,
    compute_metrics=compute_metrics
)

## Step 10: Train the Model

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.654025,0.700000,0.693878,0.680000,0.708333


Checkpoint destination directory ./results/checkpoint-63 already exists and is non-empty.Saving will proceed but saved results may be invalid.


TrainOutput(global_step=63, training_loss=0.6881357223268539, metrics={'train_runtime': 20.0812, 'train_samples_per_second': 24.899, 'train_steps_per_second': 3.137, 'total_flos': 32888881920000.0, 'train_loss': 0.6881357223268539, 'epoch': 1.0})

## Step 11: Evaluate the Model

In [13]:
predictions = trainer.predict(small_test_dataset)

print(predictions.metrics)

{'test_loss': 0.6540248394012451, 'test_accuracy': 0.7, 'test_f1': 0.6938775510204082, 'test_precision': 0.68, 'test_recall': 0.7083333333333334, 'test_runtime': 1.511, 'test_samples_per_second': 132.363, 'test_steps_per_second': 16.545}


## Step 12: Save the Fine-Tuned Model

In [14]:
trainer.save_model("bert-imdb-model")

tokenizer.save_pretrained("bert-imdb-model")

('bert-imdb-model/tokenizer_config.json',
 'bert-imdb-model/special_tokens_map.json',
 'bert-imdb-model/vocab.txt',
 'bert-imdb-model/added_tokens.json',
 'bert-imdb-model/tokenizer.json')